# Orquestador Gold — Dimensiones

Ejecuta los 5 notebooks de dimensiones **en paralelo** (son independientes entre si).
**Prerequisito:** Silver debe estar poblado (`run_silver` completado exitosamente).

| # | Notebook | Descripcion |
|---|---|---|
| 1 | `gold_dim_tiempo` | Catalogo sintetico 2012-2025 |
| 2 | `gold_dim_fuente` | UNION source_system+source_file de 10 tablas Silver |
| 3 | `gold_dim_causa` | UNION (codigo_origen, descripcion) de todas las fuentes |
| 4 | `gold_dim_geografia` | Niveles pais/departamento/municipio + placeholder INE-edad |
| 5 | `gold_dim_demografia` | Combinaciones (sexo, grupo_etario) con rangos etarios |

> **Idempotente:** todos los notebooks usan `mode('overwrite')`.
> **Timeout por notebook:** 1800 s (30 min).
> **Salida:** `'OK'` si todos completaron, `'ERRORS'` si alguno fallo.

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_this_path = _ctx.notebookPath().get()
BASE = '/'.join(_this_path.split('/')[:-2])

TIMEOUT_SEC = 1800

_NOTEBOOKS = [
    ('dim_tiempo',    f'{BASE}/transformation-scripts/gold/gold_dim_tiempo'),
    ('dim_fuente',    f'{BASE}/transformation-scripts/gold/gold_dim_fuente'),
    ('dim_causa',     f'{BASE}/transformation-scripts/gold/gold_dim_causa'),
    ('dim_geografia', f'{BASE}/transformation-scripts/gold/gold_dim_geografia'),
    ('dim_demografia',f'{BASE}/transformation-scripts/gold/gold_dim_demografia'),
]

print(f'Raiz del repo: {BASE}')
print(f'Notebooks a ejecutar en paralelo: {len(_NOTEBOOKS)}')

In [0]:
def run_notebook(args):
    label, path = args
    t0 = time.time()
    try:
        output = dbutils.notebook.run(path, TIMEOUT_SEC)
        return (label, 'OK', time.time() - t0, output or '')
    except Exception as e:
        return (label, f'ERROR: {e}', time.time() - t0, '')

results = []
t_global = time.time()

print('Lanzando 5 notebooks en paralelo...')
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(run_notebook, nb): nb[0] for nb in _NOTEBOOKS}
    for future in as_completed(futures):
        label, status, elapsed, output = future.result()
        icon = 'OK' if status == 'OK' else 'ERROR'
        print(f'  [{icon}] {label:<15} {elapsed:.0f}s  {status if status != "OK" else ""}')
        if output:
            print(f'        salida: {output}')
        results.append((label, status, elapsed))

total_elapsed = time.time() - t_global

print(f'\n{"="*60}')
print('RESUMEN GOLD DIMS')
print(f'{"="*60}')
all_ok = True
for label, status, elapsed in sorted(results, key=lambda x: x[0]):
    icon = 'OK' if status == 'OK' else 'ERROR'
    print(f'  [{icon}] {label:<15} {elapsed:>6.0f}s  {"" if status == "OK" else status}')
    if status != 'OK':
        all_ok = False

print(f'\nTiempo total (paralelo): {total_elapsed:.0f}s')
print(f'Estado final: {"TODOS OK" if all_ok else "HAY ERRORES — revisa los notebooks individuales"}')
dbutils.notebook.exit('OK' if all_ok else 'ERRORS')